# Basics: Data Loading & GPS Visualization

This notebook demonstrates the fundamentals of loading and visualizing AIM™ telemetry data using libxrk.

## What You'll Find Here

- **Data Loading**: Load `.xrk`, `.xrz`, or `.ibt` files without AIM software
- **Lap Times Table**: View all recorded laps with computed lap times
- **GPS Speed Map**: Visualize speed around the track on an interactive map
- **Brake & Throttle Overlay**: See driver inputs overlaid on the track map

## Using Your Own Data

To analyze your own data:

1. **Run the first cell** below to install packages and display the upload widget
2. **Click "Choose File"** to select your `.xrk`, `.xrz`, or `.ibt` file
3. **Run all remaining cells** to analyze your data

The status indicator will show which file is being used. If you don't upload a file, the sample data will be used.

## Requirements

- GPS data channels (`GPS Latitude`, `GPS Longitude`, `GPS Speed`)
- Brake pressure (`BrakePress`) and throttle (`PPS`) for the inputs overlay

**Note:** This notebook works in both JupyterLite (browser) and standard JupyterLab environments.

In [1]:
# Install required packages (needed for JupyterLite, skipped in regular JupyterLab if already installed)
%pip install -q motorsports-data-notebook

# Use the Rust parser backend for ~3x faster file loading
import os

os.environ["LIBXRK_BACKEND"] = "rust"

# Import helper functions
from motorsports_data_notebook.visualization import (
    format_lap_time,
    plot_gps_channels,
    show_fig,
)
from motorsports_data_notebook.widgets import SessionPicker

# Session picker with channel configuration
# Upload your own file and select a lap to analyze
session = SessionPicker(
    default_file="../data/CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz",
    channel_mapping={
        "gps_latitude": "GPS Latitude",
        "gps_longitude": "GPS Longitude",
        "throttle": "PPS",
        "brake": "BrakePress",
    },
)
session.display()

/home/runner/work/motorsports_data_notebook/motorsports_data_notebook/.venv/bin/python3: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Get laps as pandas DataFrame for display
laps = session.get_laps()

In [3]:
# Display lap times table
laps.style.format({"lap_time": format_lap_time})  # type: ignore[dict-item]

,num,start_time,end_time,lap_type,lap_time
0,1,150454,279602,full,2:09.148
1,2,279602,406240,full,2:06.638
2,3,406240,532797,full,2:06.557
3,4,532797,659283,full,2:06.486
4,5,659283,787773,full,2:08.490
5,6,787773,913776,full,2:06.003
6,7,913776,1041398,full,2:07.622
7,8,1041398,1168323,full,2:06.925
8,9,1168323,1294676,full,2:06.353
9,10,1294676,1420573,full,2:05.897


In [4]:
# Extract channel data for the selected lap using libxrk 0.5.0 methods
selected_lap = session.get_selected_lap()
log = session.get_log()
CHANNEL_NAMES = session.get_channel_names()
lap_num = int(selected_lap["num"])

# Get channel names from configuration
gps_lat_ch = CHANNEL_NAMES["gps_latitude"]
gps_lon_ch = CHANNEL_NAMES["gps_longitude"]
throttle_ch = CHANNEL_NAMES["throttle"]
brake_ch = CHANNEL_NAMES["brake"]

# Filter to lap, select channels, and resample to GPS timebase
channels = (
    log.filter_by_lap(lap_num)
    .select_channels([gps_lat_ch, gps_lon_ch, "speed_kmh", brake_ch, throttle_ch])
    .resample_to_channel(gps_lat_ch)
    .channels
)

In [5]:
# Plot speed on GPS map using channel tables directly
fig = plot_gps_channels(
    channels,
    lat_channel=gps_lat_ch,
    lon_channel=gps_lon_ch,
    color_channels=[("speed_kmh", "Speed (km/h)", "Viridis")],
    title=f"Speed - Lap {int(selected_lap['num'])}",
)
show_fig(fig)

In [6]:
# Plot with multiple color channels (automatically interpolated to GPS timebase)
fig = plot_gps_channels(
    channels,
    lat_channel=gps_lat_ch,
    lon_channel=gps_lon_ch,
    color_channels=[
        (brake_ch, "BrakePres", "Reds"),
        (throttle_ch, "Throttle", "Greens"),
    ],
    title=f"Accelerator and Brake Pressure - Lap {int(selected_lap['num'])}",
)
show_fig(fig)